# Calibration from scratch: reliability table, ECE, temperature scaling

This notebook matches notes.md §3. It builds a model whose probabilities are
overconfident, measures the miscalibration with a reliability table, the expected
calibration error (ECE), the negative log-likelihood (NLL, i.e. log loss) and the Brier
score, then fixes it with temperature scaling fit by golden-section search. Everything
uses numpy only.

Synthetic setup: each example has a true probability $p_i$ of being positive, and its
label is drawn as $y_i \sim \text{Bernoulli}(p_i)$. A calibrated model would report
$p_i$. The model here reports $\sigma(k \cdot \text{logit}(p_i))$ with $k = 3$: the
ranking of examples is unchanged (a strictly increasing transform, so AUC is unchanged),
but every probability is pushed toward 0 or 1. Dividing the logits by a temperature
$T = 3$ would undo the distortion exactly, so the fitted $T$ should come out near 3.

Run: `uv run jupyter execute ml/05_evaluation/calibration_scratch.ipynb`

In [1]:
import numpy as np

np.random.seed(42)

N = 20000
OVERCONFIDENCE = 3.0  # logit multiplier of the miscalibrated model
N_BINS = 10
EPS = 1e-12

## 1. Generate the data and split it

The true probabilities come from a Beta(2, 2) distribution, so they spread over (0, 1)
with most mass near the middle. The model's logits are 3 times the true logits. The
examples are split in half: temperature is fit on the validation half and every number
is reported on the test half, because post-hoc calibration must be fit on held-out data,
never on the training data.

In [2]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def logit(p):
    p = np.clip(p, EPS, 1 - EPS)
    return np.log(p / (1 - p))


def make_data(rng):
    p_true = rng.beta(2.0, 2.0, N)              # true P(y=1|x), spread over (0,1)
    y = (rng.uniform(size=N) < p_true).astype(int)
    z_model = OVERCONFIDENCE * logit(p_true)    # overconfident logits
    p_model = sigmoid(z_model)
    return y, z_model, p_model


rng = np.random.default_rng(42)
y, z, p = make_data(rng)

idx = rng.permutation(N)
val, test = idx[: N // 2], idx[N // 2:]
print(f"{N} examples: {len(val)} validation, {len(test)} test; "
      f"positive rate {y.mean():.3f}")

20000 examples: 10000 validation, 10000 test; positive rate 0.501


## 2. Measure the miscalibration

`reliability_table` bins the test predictions into 10 equal-width bins of predicted
$P(y=1)$ and compares, in each bin, the mean predicted probability with the observed
fraction of positives. ECE is the bin-count-weighted average of the gaps. For an
overconfident model, expect bins below 0.5 to have a higher positive rate than predicted
and bins above 0.5 a lower one, with the gaps generally larger farther from 0.5.

In [3]:
def reliability_table(y, p, n_bins=N_BINS, label=""):
    """Bin by predicted probability; compare mean confidence vs empirical accuracy.

    Uses P(y=1) directly (binary), i.e. bins of predicted positive-probability
    vs observed positive rate. Returns ECE.
    """
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    which = np.clip(np.digitize(p, edges[1:-1]), 0, n_bins - 1)

    print(f"Reliability table {label} (bins of predicted P(y=1)):")
    header = f"{'bin':>12} {'count':>7} {'mean conf':>10} {'frac pos':>9} {'|gap|':>8}"
    print(header)
    print("-" * len(header))
    ece = 0.0
    for b in range(n_bins):
        mask = which == b
        n_b = int(mask.sum())
        if n_b == 0:
            print(f"[{edges[b]:.1f}, {edges[b+1]:.1f}) {'-':>7}")
            continue
        conf = float(p[mask].mean())
        acc = float(y[mask].mean())
        gap = abs(acc - conf)
        ece += (n_b / len(y)) * gap
        print(f"[{edges[b]:.1f}, {edges[b+1]:.1f}) {n_b:>7} {conf:>10.4f} {acc:>9.4f} {gap:>8.4f}")
    print("-" * len(header))
    print(f"ECE {label} = {ece:.4f}")
    return ece


def nll(y, p):
    p = np.clip(p, EPS, 1 - EPS)
    return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))


def brier(y, p):
    return float(np.mean((p - y) ** 2))


print(f"Overconfident model: logits multiplied by k={OVERCONFIDENCE}")
print()
ece_before = reliability_table(y[test], p[test], label="BEFORE")
nll_before, brier_before = nll(y[test], p[test]), brier(y[test], p[test])
print(f"NLL before   = {nll_before:.4f}")
print(f"Brier before = {brier_before:.4f}")

Overconfident model: logits multiplied by k=3.0

Reliability table BEFORE (bins of predicted P(y=1)):
         bin   count  mean conf  frac pos    |gap|
--------------------------------------------------
[0.0, 0.1)    2431     0.0310    0.1937   0.1628
[0.1, 0.2)     865     0.1456    0.3630   0.2174
[0.2, 0.3)     619     0.2485    0.3845   0.1360
[0.3, 0.4)     545     0.3482    0.4330   0.0848
[0.4, 0.5)     511     0.4502    0.4716   0.0215
[0.5, 0.6)     521     0.5485    0.4837   0.0648
[0.6, 0.7)     568     0.6520    0.5299   0.1221
[0.7, 0.8)     636     0.7520    0.5912   0.1608
[0.8, 0.9)     817     0.8539    0.6781   0.1758
[0.9, 1.0)    2487     0.9687    0.7961   0.1726
--------------------------------------------------
ECE BEFORE = 0.1503
NLL before   = 0.7444
Brier before = 0.2232


## 3. Fit the temperature and re-measure

Temperature scaling replaces each logit $z$ with $z / T$ and picks the single number $T$
that minimizes validation NLL. NLL as a function of $T$ has one minimum here, so
golden-section search finds it without gradients: it keeps a bracket $[a, b]$ around the
minimum and shrinks it by a factor of about 0.618 per step. Expect $T$ close to 3, and
ECE, NLL and Brier on the test half all lower than before.

In [4]:
def fit_temperature(y, z, lo=0.05, hi=20.0, tol=1e-6):
    """Golden-section search for T minimizing validation NLL of sigmoid(z / T).

    NLL as a function of T is unimodal here; golden-section needs no gradients
    and no scipy.
    """
    invphi = (np.sqrt(5.0) - 1.0) / 2.0

    def obj(t):
        return nll(y, sigmoid(z / t))

    a, b = lo, hi
    c, d = b - invphi * (b - a), a + invphi * (b - a)
    fc, fd = obj(c), obj(d)
    while b - a > tol:
        if fc < fd:
            b, d, fd = d, c, fc
            c = b - invphi * (b - a)
            fc = obj(c)
        else:
            a, c, fc = c, d, fd
            d = a + invphi * (b - a)
            fd = obj(d)
    return 0.5 * (a + b)


T = fit_temperature(y[val], z[val])
p_cal = sigmoid(z / T)
print(f"Fitted temperature on validation half: T = {T:.4f} "
      f"(true distortion factor was {OVERCONFIDENCE})")
print()
ece_after = reliability_table(y[test], p_cal[test], label="AFTER")
nll_after, brier_after = nll(y[test], p_cal[test]), brier(y[test], p_cal[test])
print(f"NLL after   = {nll_after:.4f}")
print(f"Brier after = {brier_after:.4f}")
print()
print(f"Summary: ECE {ece_before:.4f} -> {ece_after:.4f}, "
      f"NLL {nll_before:.4f} -> {nll_after:.4f}, Brier {brier_before:.4f} -> {brier_after:.4f}")
assert ece_after < ece_before

Fitted temperature on validation half: T = 3.0890 (true distortion factor was 3.0)

Reliability table AFTER (bins of predicted P(y=1)):
         bin   count  mean conf  frac pos    |gap|
--------------------------------------------------
[0.0, 0.1)     231     0.0650    0.0476   0.0174
[0.1, 0.2)     727     0.1546    0.1431   0.0115
[0.2, 0.3)    1117     0.2528    0.2319   0.0209
[0.3, 0.4)    1376     0.3518    0.3401   0.0117
[0.4, 0.5)    1520     0.4508    0.4329   0.0179
[0.5, 0.6)    1581     0.5498    0.5294   0.0204
[0.6, 0.7)    1356     0.6497    0.6844   0.0347
[0.7, 0.8)    1133     0.7491    0.7476   0.0015
[0.8, 0.9)     708     0.8457    0.8672   0.0216
[0.9, 1.0)     251     0.9356    0.9442   0.0086
--------------------------------------------------
ECE AFTER = 0.0177
NLL after   = 0.5775
Brier after = 0.1974

Summary: ECE 0.1503 -> 0.0177, NLL 0.7444 -> 0.5775, Brier 0.2232 -> 0.1974


## 4. AUC is unchanged

Dividing every logit by the same positive $T$ keeps the order of the examples, and AUC
depends only on that order (notes.md §2). The cell computes AUC before and after
calibration from rank sums. Expect the two values to be identical to within 1e-12.

In [5]:
def auc_rank(y, s):
    order = np.argsort(s, kind="stable")
    ranks = np.empty(len(s))
    ranks[order] = np.arange(1, len(s) + 1)  # no ties expected (continuous scores)
    n_pos = int(y.sum())
    n_neg = len(y) - n_pos
    u = ranks[y == 1].sum() - n_pos * (n_pos + 1) / 2
    return float(u / (n_pos * n_neg))


auc_b = auc_rank(y[test], p[test])
auc_a = auc_rank(y[test], p_cal[test])
print(f"AUC before = {auc_b:.6f}, AUC after = {auc_a:.6f} "
      f"(identical: z/T is a monotone transform)")
assert abs(auc_b - auc_a) < 1e-12

AUC before = 0.764542, AUC after = 0.764542 (identical: z/T is a monotone transform)
